# WorldQuant Alpha101 Factor Implementation

Implementation of 101 formulaic alphas from "101 Formulaic Alphas" by Zura Kakushadze (2015).

Reference: https://arxiv.org/abs/1601.00991

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')

## Helper Functions and Operators

These functions implement the operators defined in Appendix A.2 of the paper.

In [ ]:
# Basic operators
def rank(x):
    """
    Cross-sectional rank
    """
    if isinstance(x, pd.DataFrame):
        return x.rank(axis=1, pct=True)
    elif isinstance(x, pd.Series):
        return x.rank(pct=True)
    else:
        return pd.Series(x).rank(pct=True)

def delay(x, d):
    """
    Value of x d days ago
    """
    return x.shift(d)

def correlation(x, y, d):
    """
    Time-serial correlation of x and y for the past d days
    """
    return x.rolling(window=d).corr(y)

def covariance(x, y, d):
    """
    Time-serial covariance of x and y for the past d days
    """
    return x.rolling(window=d).cov(y)

def scale(x, a=1):
    """
    Rescaled x such that sum(abs(x)) = a (default is a = 1)
    """
    if isinstance(x, pd.DataFrame):
        return x.div(x.abs().sum(axis=1), axis=0) * a
    else:
        return x / x.abs().sum() * a

def delta(x, d):
    """
    Today's value of x minus the value of x d days ago
    """
    return x.diff(d)

def signedpower(x, a):
    """
    Signed power: x^a
    """
    return np.sign(x) * (np.abs(x) ** a)

def decay_linear(x, d):
    """
    Weighted moving average over the past d days with linearly decaying weights d, d-1, ..., 1
    """
    if isinstance(x, pd.DataFrame):
        weights = np.arange(1, d + 1)
        weights = weights / weights.sum()
        return x.rolling(window=d).apply(lambda x: (x * weights).sum(), raw=True)
    else:
        weights = np.arange(1, d + 1)
        weights = weights / weights.sum()
        return x.rolling(window=d).apply(lambda x: (x * weights).sum(), raw=True)

def indneutralize(x, g):
    """
    Cross-sectionally neutralized x against groups g
    x is cross-sectionally demeaned within each group g
    """
    if isinstance(x, pd.DataFrame):
        return x.subtract(x.groupby(g, axis=1).transform('mean'), axis=0)
    else:
        return x - x.groupby(g).transform('mean')

# Time-series operators
def ts_min(x, d):
    """
    Time-series min over the past d days
    """
    return x.rolling(window=d).min()

def ts_max(x, d):
    """
    Time-series max over the past d days
    """
    return x.rolling(window=d).max()

def ts_argmax(x, d):
    """
    Which day ts_max(x, d) occurred on
    """
    return x.rolling(window=d).apply(lambda x: x.argmax() + 1, raw=True)

def ts_argmin(x, d):
    """
    Which day ts_min(x, d) occurred on
    """
    return x.rolling(window=d).apply(lambda x: x.argmin() + 1, raw=True)

def ts_rank(x, d):
    """
    Time-series rank in the past d days
    """
    return x.rolling(window=d).apply(lambda x: rankdata(x)[-1] / len(x), raw=True)

def sum(x, d):
    """
    Time-series sum over the past d days
    """
    return x.rolling(window=d).sum()

def product(x, d):
    """
    Time-series product over the past d days
    """
    return x.rolling(window=d).apply(lambda x: np.prod(x), raw=True)

def stddev(x, d):
    """
    Moving time-series standard deviation over the past d days
    """
    return x.rolling(window=d).std()

## Alpha Factor Implementations

All 101 alpha factors from the paper. Each function expects a DataFrame with columns:
- open, high, low, close, volume, vwap, returns
- Optional: cap (market cap), industry classification for neutralization

In [ ]:
def alpha001(data):
    """
    (rank(Ts_ArgMax(SignedPower(((returns < 0) ? stddev(returns, 20) : close), 2.), 5)) - 0.5)
    """
    condition = data['returns'] < 0
    part1 = condition * stddev(data['returns'], 20) + (~condition) * data['close']
    return rank(ts_argmax(signedpower(part1, 2), 5)) - 0.5

def alpha002(data):
    """
    (-1 * correlation(rank(delta(log(volume), 2)), rank(((close - open) / open)), 6))
    """
    return -1 * correlation(rank(delta(np.log(data['volume']), 2)), 
                           rank((data['close'] - data['open']) / data['open']), 6)

def alpha003(data):
    """
    (-1 * correlation(rank(open), rank(volume), 10))
    """
    return -1 * correlation(rank(data['open']), rank(data['volume']), 10)

def alpha004(data):
    """
    (-1 * Ts_Rank(rank(low), 9))
    """
    return -1 * ts_rank(rank(data['low']), 9)

def alpha005(data):
    """
    (rank((open - (sum(vwap, 10) / 10))) * (-1 * abs(rank((close - vwap)))))
    """
    return rank(data['open'] - sum(data['vwap'], 10) / 10) * (-1 * np.abs(rank(data['close'] - data['vwap'])))

def alpha006(data):
    """
    (-1 * correlation(open, volume, 10))
    """
    return -1 * correlation(data['open'], data['volume'], 10)

def alpha007(data):
    """
    ((adv20 < volume) ? ((-1 * ts_rank(abs(delta(close, 7)), 60)) * sign(delta(close, 7))) : (-1 * 1))
    """
    adv20 = sum(data['volume'], 20) / 20
    condition = adv20 < data['volume']
    alpha = -1 * ts_rank(np.abs(delta(data['close'], 7)), 60) * np.sign(delta(data['close'], 7))
    return condition * alpha + (~condition) * (-1)

def alpha008(data):
    """
    (-1 * rank(((sum(open, 5) * sum(returns, 5)) - delay((sum(open, 5) * sum(returns, 5)), 10))))
    """
    return -1 * rank(sum(data['open'], 5) * sum(data['returns'], 5) - 
                     delay(sum(data['open'], 5) * sum(data['returns'], 5), 10))

def alpha009(data):
    """
    ((0 < ts_min(delta(close, 1), 5)) ? delta(close, 1) : 
     ((ts_max(delta(close, 1), 5) < 0) ? delta(close, 1) : (-1 * delta(close, 1))))
    """
    delta_close = delta(data['close'], 1)
    condition1 = ts_min(delta_close, 5) > 0
    condition2 = ts_max(delta_close, 5) < 0
    return condition1 * delta_close + (~condition1) * (condition2 * delta_close + (~condition2) * (-1 * delta_close))

def alpha010(data):
    """
    rank(((0 < ts_min(delta(close, 1), 4)) ? delta(close, 1) : 
          ((ts_max(delta(close, 1), 4) < 0) ? delta(close, 1) : (-1 * delta(close, 1)))))
    """
    delta_close = delta(data['close'], 1)
    condition1 = ts_min(delta_close, 4) > 0
    condition2 = ts_max(delta_close, 4) < 0
    return rank(condition1 * delta_close + (~condition1) * (condition2 * delta_close + (~condition2) * (-1 * delta_close)))

def alpha011(data):
    """
    ((rank(ts_max((vwap - close), 3)) + rank(ts_min((vwap - close), 3))) * rank(delta(volume, 3)))
    """
    return (rank(ts_max(data['vwap'] - data['close'], 3)) + 
            rank(ts_min(data['vwap'] - data['close'], 3))) * rank(delta(data['volume'], 3))

def alpha012(data):
    """
    (sign(delta(volume, 1)) * (-1 * delta(close, 1)))
    """
    return np.sign(delta(data['volume'], 1)) * (-1 * delta(data['close'], 1))

def alpha013(data):
    """
    (-1 * rank(covariance(rank(close), rank(volume), 5)))
    """
    return -1 * rank(covariance(rank(data['close']), rank(data['volume']), 5))

def alpha014(data):
    """
    ((-1 * rank(delta(returns, 3))) * correlation(open, volume, 10))
    """
    return -1 * rank(delta(data['returns'], 3)) * correlation(data['open'], data['volume'], 10)

def alpha015(data):
    """
    (-1 * sum(rank(correlation(rank(high), rank(volume), 3)), 3))
    """
    return -1 * sum(rank(correlation(rank(data['high']), rank(data['volume']), 3)), 3)

def alpha016(data):
    """
    (-1 * rank(covariance(rank(high), rank(volume), 5)))
    """
    return -1 * rank(covariance(rank(data['high']), rank(data['volume']), 5))

def alpha017(data):
    """
    (((-1 * rank(ts_rank(close, 10))) * rank(delta(delta(close, 1), 1))) * 
     rank(ts_rank((volume / adv20), 5)))
    """
    adv20 = sum(data['volume'], 20) / 20
    return ((-1 * rank(ts_rank(data['close'], 10))) * rank(delta(delta(data['close'], 1), 1)) * 
            rank(ts_rank(data['volume'] / adv20, 5)))

def alpha018(data):
    """
    (-1 * rank(((stddev(abs((close - open)), 5) + (close - open)) + correlation(close, open, 10))))
    """
    return -1 * rank(stddev(np.abs(data['close'] - data['open']), 5) + 
                     (data['close'] - data['open']) + correlation(data['close'], data['open'], 10))

def alpha019(data):
    """
    ((-1 * sign(((close - delay(close, 7)) + delta(close, 7)))) * (1 + rank((1 + sum(returns, 250)))))
    """
    return (-1 * np.sign((data['close'] - delay(data['close'], 7)) + delta(data['close'], 7))) * \
           (1 + rank(1 + sum(data['returns'], 250)))

def alpha020(data):
    """
    (((-1 * rank((open - delay(high, 1)))) * rank((open - delay(close, 1)))) * rank((open - delay(low, 1))))
    """
    return ((-1 * rank(data['open'] - delay(data['high'], 1))) * 
            rank(data['open'] - delay(data['close'], 1)) * 
            rank(data['open'] - delay(data['low'], 1)))

def alpha021(data):
    """
    ((((sum(close, 8) / 8) + stddev(close, 8)) < (sum(close, 2) / 2)) ? (-1 * 1) : 
     (((sum(close, 2) / 2) < ((sum(close, 8) / 8) - stddev(close, 8))) ? 1 : 
      (((1 < (volume / adv20)) || ((volume / adv20) == 1)) ? 1 : (-1 * 1))))
    """
    adv20 = sum(data['volume'], 20) / 20
    condition1 = (sum(data['close'], 8) / 8 + stddev(data['close'], 8)) < (sum(data['close'], 2) / 2)
    condition2 = (sum(data['close'], 2) / 2) < (sum(data['close'], 8) / 8 - stddev(data['close'], 8))
    condition3 = (data['volume'] / adv20) >= 1
    return condition1 * (-1) + (~condition1) * (condition2 * 1 + (~condition2) * (condition3 * 1 + (~condition3) * (-1)))

def alpha022(data):
    """
    (-1 * (delta(correlation(high, volume, 5), 5) * rank(stddev(close, 20))))
    """
    return -1 * delta(correlation(data['high'], data['volume'], 5), 5) * rank(stddev(data['close'], 20))

def alpha023(data):
    """
    (((sum(high, 20) / 20) < high) ? (-1 * delta(high, 2)) : 0)
    """
    condition = (sum(data['high'], 20) / 20) < data['high']
    return condition * (-1 * delta(data['high'], 2))

def alpha024(data):
    """
    ((((delta((sum(close, 100) / 100), 100) / delay(close, 100)) < 0.05) ||
      ((delta((sum(close, 100) / 100), 100) / delay(close, 100)) == 0.05)) ? 
     (-1 * (close - ts_min(close, 100))) : (-1 * delta(close, 3)))
    """
    condition = (delta(sum(data['close'], 100) / 100, 100) / delay(data['close'], 100)) <= 0.05
    return condition * (-1 * (data['close'] - ts_min(data['close'], 100))) + \
           (~condition) * (-1 * delta(data['close'], 3))

def alpha025(data):
    """
    rank(((((-1 * returns) * adv20) * vwap) * (high - close)))
    """
    adv20 = sum(data['volume'], 20) / 20
    return rank(((-1 * data['returns']) * adv20 * data['vwap'] * (data['high'] - data['close'])))

def alpha026(data):
    """
    (-1 * ts_max(correlation(ts_rank(volume, 5), ts_rank(high, 5), 5), 3))
    """
    return -1 * ts_max(correlation(ts_rank(data['volume'], 5), ts_rank(data['high'], 5), 5), 3)

def alpha027(data):
    """
    ((0.5 < rank((sum(correlation(rank(volume), rank(vwap), 6), 2) / 2.0))) ? (-1 * 1) : 1)
    """
    condition = rank(sum(correlation(rank(data['volume']), rank(data['vwap']), 6), 2) / 2.0) > 0.5
    return condition * (-1) + (~condition) * 1

def alpha028(data):
    """
    scale(((correlation(adv20, low, 5) + ((high + low) / 2)) - close))
    """
    adv20 = sum(data['volume'], 20) / 20
    return scale(correlation(adv20, data['low'], 5) + (data['high'] + data['low']) / 2 - data['close'])

def alpha029(data):
    """
    (min(product(rank(rank(scale(log(sum(ts_min(rank(rank((-1 * rank(delta((close - 1), 5))))), 2), 1))))), 1), 5) +
     ts_rank(delay((-1 * returns), 6), 5))
    """
    part1 = product(rank(rank(scale(np.log(sum(ts_min(rank(rank(-1 * rank(delta(data['close'] - 1, 5)))), 2), 1))))), 1)
    return ts_min(part1, 5) + ts_rank(delay(-1 * data['returns'], 6), 5)

def alpha030(data):
    """
    (((1.0 - rank(((sign((close - delay(close, 1))) + sign((delay(close, 1) - delay(close, 2)))) +
      sign((delay(close, 2) - delay(close, 3)))))) * sum(volume, 5)) / sum(volume, 20))
    """
    sign_sum = (np.sign(data['close'] - delay(data['close'], 1)) + 
                np.sign(delay(data['close'], 1) - delay(data['close'], 2)) + 
                np.sign(delay(data['close'], 2) - delay(data['close'], 3)))
    return (1.0 - rank(sign_sum)) * sum(data['volume'], 5) / sum(data['volume'], 20)

def alpha031(data):
    """
    ((rank(rank(rank(decay_linear((-1 * rank(rank(delta(close, 10)))), 10)))) + rank((-1 * delta(close, 3)))) +
     sign(scale(correlation(adv20, low, 12))))
    """
    adv20 = sum(data['volume'], 20) / 20
    return (rank(rank(rank(decay_linear(-1 * rank(rank(delta(data['close'], 10))), 10)))) + 
            rank(-1 * delta(data['close'], 3)) + 
            np.sign(scale(correlation(adv20, data['low'], 12))))

def alpha032(data):
    """
    (scale(((sum(close, 7) / 7) - close)) + (20 * scale(correlation(vwap, delay(close, 5), 230))))
    """
    return scale(sum(data['close'], 7) / 7 - data['close']) + \
           20 * scale(correlation(data['vwap'], delay(data['close'], 5), 230))

def alpha033(data):
    """
    rank((-1 * ((1 - (open / close))^1)))
    """
    return rank(-1 * (1 - data['open'] / data['close']))

def alpha034(data):
    """
    rank(((1 - rank((stddev(returns, 2) / stddev(returns, 5)))) + (1 - rank(delta(close, 1)))))
    """
    return rank((1 - rank(stddev(data['returns'], 2) / stddev(data['returns'], 5))) + 
                (1 - rank(delta(data['close'], 1))))

def alpha035(data):
    """
    ((Ts_Rank(volume, 32) * (1 - Ts_Rank(((close + high) - low), 16))) * (1 - Ts_Rank(returns, 32)))
    """
    return (ts_rank(data['volume'], 32) * 
            (1 - ts_rank(data['close'] + data['high'] - data['low'], 16)) * 
            (1 - ts_rank(data['returns'], 32)))

def alpha036(data):
    """
    (((((2.21 * rank(correlation((close - open), delay(volume, 1), 15))) + (0.7 * rank((open - close)))) +
      (0.73 * rank(Ts_Rank(delay((-1 * returns), 6), 5)))) + rank(abs(correlation(vwap, adv20, 6)))) +
     (0.6 * rank((((sum(close, 200) / 200) - open) * (close - open)))))
    """
    adv20 = sum(data['volume'], 20) / 20
    return (2.21 * rank(correlation(data['close'] - data['open'], delay(data['volume'], 1), 15)) +
            0.7 * rank(data['open'] - data['close']) +
            0.73 * rank(ts_rank(delay(-1 * data['returns'], 6), 5)) +
            rank(np.abs(correlation(data['vwap'], adv20, 6))) +
            0.6 * rank((sum(data['close'], 200) / 200 - data['open']) * (data['close'] - data['open'])))

def alpha037(data):
    """
    (rank(correlation(delay((open - close), 1), close, 200)) + rank((open - close)))
    """
    return rank(correlation(delay(data['open'] - data['close'], 1), data['close'], 200)) + \
           rank(data['open'] - data['close'])

def alpha038(data):
    """
    ((-1 * rank(Ts_Rank(close, 10))) * rank((close / open)))
    """
    return -1 * rank(ts_rank(data['close'], 10)) * rank(data['close'] / data['open'])

def alpha039(data):
    """
    ((-1 * rank((delta(close, 7) * (1 - rank(decay_linear((volume / adv20), 9)))))) *
     (1 + rank(sum(returns, 250))))
    """
    adv20 = sum(data['volume'], 20) / 20
    return (-1 * rank(delta(data['close'], 7) * (1 - rank(decay_linear(data['volume'] / adv20, 9))))) * \
           (1 + rank(sum(data['returns'], 250)))

def alpha040(data):
    """
    ((-1 * rank(stddev(high, 10))) * correlation(high, volume, 10))
    """
    return -1 * rank(stddev(data['high'], 10)) * correlation(data['high'], data['volume'], 10)

def alpha041(data):
    """
    (((high * low)^0.5) - vwap)
    """
    return (data['high'] * data['low']) ** 0.5 - data['vwap']

def alpha042(data):
    """
    (rank((vwap - close)) / rank((vwap + close)))
    """
    return rank(data['vwap'] - data['close']) / rank(data['vwap'] + data['close'])

def alpha043(data):
    """
    (ts_rank((volume / adv20), 20) * ts_rank((-1 * delta(close, 7)), 8))
    """
    adv20 = sum(data['volume'], 20) / 20
    return ts_rank(data['volume'] / adv20, 20) * ts_rank(-1 * delta(data['close'], 7), 8)

def alpha044(data):
    """
    (-1 * correlation(high, rank(volume), 5))
    """
    return -1 * correlation(data['high'], rank(data['volume']), 5)

def alpha045(data):
    """
    (-1 * ((rank((sum(delay(close, 5), 20) / 20)) * correlation(close, volume, 2)) *
     rank(correlation(sum(close, 5), sum(close, 20), 2))))
    """
    return -1 * (rank(sum(delay(data['close'], 5), 20) / 20) * 
                 correlation(data['close'], data['volume'], 2) *
                 rank(correlation(sum(data['close'], 5), sum(data['close'], 20), 2)))

def alpha046(data):
    """
    ((0.25 < (((delay(close, 20) - delay(close, 10)) / 10) - ((delay(close, 10) - close) / 10))) ?
     (-1 * 1) : (((((delay(close, 20) - delay(close, 10)) / 10) - ((delay(close, 10) - close) / 10)) < 0) ? 1 :
     ((-1 * 1) * (close - delay(close, 1)))))
    """
    inner = (delay(data['close'], 20) - delay(data['close'], 10)) / 10 - \
            (delay(data['close'], 10) - data['close']) / 10
    condition1 = inner > 0.25
    condition2 = inner < 0
    return condition1 * (-1) + (~condition1) * (condition2 * 1 + (~condition2) * (-1 * (data['close'] - delay(data['close'], 1))))

def alpha047(data):
    """
    ((((rank((1 / close)) * volume) / adv20) * ((high * rank((high - close))) / (sum(high, 5) / 5))) -
     rank((vwap - delay(vwap, 5))))
    """
    adv20 = sum(data['volume'], 20) / 20
    return ((rank(1 / data['close']) * data['volume'] / adv20) * 
            (data['high'] * rank(data['high'] - data['close']) / (sum(data['high'], 5) / 5)) - 
            rank(data['vwap'] - delay(data['vwap'], 5)))

def alpha048(data):
    """
    (indneutralize(((correlation(delta(close, 1), delta(delay(close, 1), 1), 250) *
     delta(close, 1)) / close), IndClass.subindustry) / sum(((delta(close, 1) / delay(close, 1))^2), 250))
    
    Note: Requires industry classification data
    """
    # Simplified version without industry neutralization
    numerator = correlation(delta(data['close'], 1), delta(delay(data['close'], 1), 1), 250) * \
                delta(data['close'], 1) / data['close']
    denominator = sum((delta(data['close'], 1) / delay(data['close'], 1)) ** 2, 250)
    return numerator / denominator

def alpha049(data):
    """
    (((((delay(close, 20) - delay(close, 10)) / 10) - ((delay(close, 10) - close) / 10)) < (-1 * 0.1)) ? 1 :
     ((-1 * 1) * (close - delay(close, 1))))
    """
    inner = (delay(data['close'], 20) - delay(data['close'], 10)) / 10 - \
            (delay(data['close'], 10) - data['close']) / 10
    condition = inner < -0.1
    return condition * 1 + (~condition) * (-1 * (data['close'] - delay(data['close'], 1)))

def alpha050(data):
    """
    (-1 * ts_max(rank(correlation(rank(volume), rank(vwap), 5)), 5))
    """
    return -1 * ts_max(rank(correlation(rank(data['volume']), rank(data['vwap']), 5)), 5)

def alpha051(data):
    """
    (((((delay(close, 20) - delay(close, 10)) / 10) - ((delay(close, 10) - close) / 10)) < (-1 * 0.05)) ? 1 :
     ((-1 * 1) * (close - delay(close, 1))))
    """
    inner = (delay(data['close'], 20) - delay(data['close'], 10)) / 10 - \
            (delay(data['close'], 10) - data['close']) / 10
    condition = inner < -0.05
    return condition * 1 + (~condition) * (-1 * (data['close'] - delay(data['close'], 1)))

def alpha052(data):
    """
    ((((-1 * ts_min(low, 5)) + delay(ts_min(low, 5), 5)) *
     rank(((sum(returns, 240) - sum(returns, 20)) / 220))) * ts_rank(volume, 5))
    """
    return (((-1 * ts_min(data['low'], 5)) + delay(ts_min(data['low'], 5), 5)) *
            rank((sum(data['returns'], 240) - sum(data['returns'], 20)) / 220) *
            ts_rank(data['volume'], 5))

def alpha053(data):
    """
    (-1 * delta((((close - low) - (high - close)) / (close - low)), 9))
    """
    return -1 * delta(((data['close'] - data['low']) - (data['high'] - data['close'])) / 
                      (data['close'] - data['low']), 9)

def alpha054(data):
    """
    ((-1 * ((low - close) * (open^5))) / ((low - high) * (close^5)))
    """
    return -1 * ((data['low'] - data['close']) * (data['open'] ** 5)) / \
           ((data['low'] - data['high']) * (data['close'] ** 5))

def alpha055(data):
    """
    (-1 * correlation(rank(((close - ts_min(low, 12)) / (ts_max(high, 12) - ts_min(low, 12)))),
     rank(volume), 6))
    """
    return -1 * correlation(
        rank((data['close'] - ts_min(data['low'], 12)) / 
             (ts_max(data['high'], 12) - ts_min(data['low'], 12))),
        rank(data['volume']), 6)

def alpha056(data):
    """
    (0 - (1 * (rank((sum(returns, 10) / sum(sum(returns, 2), 3))) * rank((returns * cap)))))
    
    Note: Requires market cap data
    """
    # Simplified version without cap
    return -1 * rank(sum(data['returns'], 10) / sum(sum(data['returns'], 2), 3)) * rank(data['returns'])

def alpha057(data):
    """
    (0 - (1 * ((close - vwap) / decay_linear(rank(ts_argmax(close, 30)), 2))))
    """
    return -1 * ((data['close'] - data['vwap']) / 
                 decay_linear(rank(ts_argmax(data['close'], 30)), 2))

def alpha058(data):
    """
    (-1 * Ts_Rank(decay_linear(correlation(IndNeutralize(vwap, IndClass.sector), volume,
     3.92795), 7.89291), 5.50322))
    
    Note: Requires industry classification data
    """
    # Simplified version without industry neutralization
    return -1 * ts_rank(decay_linear(correlation(data['vwap'], data['volume'], 4), 8), 6)

def alpha059(data):
    """
    (-1 * Ts_Rank(decay_linear(correlation(IndNeutralize(((vwap * 0.728317) + (vwap * (1 - 0.728317))),
     IndClass.industry), volume, 4.25197), 16.2289), 8.19648))
    
    Note: Requires industry classification data
    """
    # Simplified version without industry neutralization
    return -1 * ts_rank(decay_linear(correlation(data['vwap'], data['volume'], 4), 16), 8)

def alpha060(data):
    """
    (0 - (1 * ((2 * scale(rank(((((close - low) - (high - close)) / (high - low)) * volume)))) -
     scale(rank(ts_argmax(close, 10))))))
    """
    return -1 * (2 * scale(rank(((data['close'] - data['low']) - (data['high'] - data['close'])) / 
                                (data['high'] - data['low']) * data['volume'])) -
                 scale(rank(ts_argmax(data['close'], 10))))

def alpha061(data):
    """
    (rank((vwap - ts_min(vwap, 16.1219))) < rank(correlation(vwap, adv180, 17.9282)))
    """
    adv180 = sum(data['volume'], 180) / 180
    return (rank(data['vwap'] - ts_min(data['vwap'], 16)) < 
            rank(correlation(data['vwap'], adv180, 18))).astype(int)

def alpha062(data):
    """
    ((rank(correlation(vwap, sum(adv20, 22.4101), 9.91009)) <
     rank(((rank(open) + rank(open)) < (rank(((high + low) / 2)) + rank(high))))) * -1)
    """
    adv20 = sum(data['volume'], 20) / 20
    return ((rank(correlation(data['vwap'], sum(adv20, 22), 10)) <
             rank((rank(data['open']) + rank(data['open'])) < 
                  (rank((data['high'] + data['low']) / 2) + rank(data['high'])))) * -1)

def alpha063(data):
    """
    ((rank(decay_linear(delta(IndNeutralize(close, IndClass.industry), 2.25164), 8.22237)) -
     rank(decay_linear(correlation(((vwap * 0.318108) + (open * (1 - 0.318108))), sum(adv180, 37.2467),
     13.557), 12.2883))) * -1)
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv180 = sum(data['volume'], 180) / 180
    return (rank(decay_linear(delta(data['close'], 2), 8)) -
            rank(decay_linear(correlation(data['vwap'] * 0.318108 + data['open'] * 0.681892,
                                         sum(adv180, 37), 14), 12))) * -1

def alpha064(data):
    """
    ((rank(correlation(sum(((open * 0.178404) + (low * (1 - 0.178404))), 12.7054),
     sum(adv120, 12.7054), 16.6208)) <
     rank(delta(((((high + low) / 2) * 0.178404) + (vwap * (1 - 0.178404))), 3.69741))) * -1)
    """
    adv120 = sum(data['volume'], 120) / 120
    return ((rank(correlation(sum(data['open'] * 0.178404 + data['low'] * 0.821596, 13),
                             sum(adv120, 13), 17)) <
             rank(delta((data['high'] + data['low']) / 2 * 0.178404 + data['vwap'] * 0.821596, 4))) * -1)

def alpha065(data):
    """
    ((rank(correlation(((open * 0.00817205) + (vwap * (1 - 0.00817205))), sum(adv60, 8.6911), 6.40374)) <
     rank((open - ts_min(open, 13.635)))) * -1)
    """
    adv60 = sum(data['volume'], 60) / 60
    return ((rank(correlation(data['open'] * 0.00817205 + data['vwap'] * 0.99182795,
                             sum(adv60, 9), 6)) <
             rank(data['open'] - ts_min(data['open'], 14))) * -1)

def alpha066(data):
    """
    ((rank(decay_linear(delta(vwap, 3.51013), 7.23052)) +
     Ts_Rank(decay_linear(((((low * 0.96633) + (low * (1 - 0.96633))) - vwap) /
     (open - ((high + low) / 2))), 11.4157), 6.72611)) * -1)
    """
    return (rank(decay_linear(delta(data['vwap'], 4), 7)) +
            ts_rank(decay_linear((data['low'] - data['vwap']) /
                                (data['open'] - (data['high'] + data['low']) / 2), 11), 7)) * -1

def alpha067(data):
    """
    ((rank((high - ts_min(high, 2.14593)))^rank(correlation(IndNeutralize(vwap, IndClass.sector),
     IndNeutralize(adv20, IndClass.subindustry), 6.02936))) * -1)
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv20 = sum(data['volume'], 20) / 20
    return (rank(data['high'] - ts_min(data['high'], 2)) ** 
            rank(correlation(data['vwap'], adv20, 6))) * -1

def alpha068(data):
    """
    ((Ts_Rank(correlation(rank(high), rank(adv15), 8.91644), 13.9333) <
     rank(delta(((close * 0.518371) + (low * (1 - 0.518371))), 1.06157))) * -1)
    """
    adv15 = sum(data['volume'], 15) / 15
    return ((ts_rank(correlation(rank(data['high']), rank(adv15), 9), 14) <
             rank(delta(data['close'] * 0.518371 + data['low'] * 0.481629, 1))) * -1)

def alpha069(data):
    """
    ((rank(ts_max(delta(IndNeutralize(vwap, IndClass.industry), 2.72412), 4.79344))^
     Ts_Rank(correlation(((close * 0.490655) + (vwap * (1 - 0.490655))), adv20, 4.92416), 9.0615)) * -1)
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv20 = sum(data['volume'], 20) / 20
    return (rank(ts_max(delta(data['vwap'], 3), 5)) **
            ts_rank(correlation(data['close'] * 0.490655 + data['vwap'] * 0.509345, adv20, 5), 9)) * -1

def alpha070(data):
    """
    ((rank(delta(vwap, 1.29456))^Ts_Rank(correlation(IndNeutralize(close, IndClass.industry),
     adv50, 17.8256), 17.9171)) * -1)
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv50 = sum(data['volume'], 50) / 50
    return (rank(delta(data['vwap'], 1)) ** 
            ts_rank(correlation(data['close'], adv50, 18), 18)) * -1

def alpha071(data):
    """
    max(Ts_Rank(decay_linear(correlation(Ts_Rank(close, 3.43976), Ts_Rank(adv180, 12.0647), 18.0175),
     4.20501), 15.6948), Ts_Rank(decay_linear((rank(((low + open) - (vwap + vwap)))^2), 16.4662), 4.4388))
    """
    adv180 = sum(data['volume'], 180) / 180
    part1 = ts_rank(decay_linear(correlation(ts_rank(data['close'], 3), ts_rank(adv180, 12), 18), 4), 16)
    part2 = ts_rank(decay_linear(rank((data['low'] + data['open']) - 2 * data['vwap']) ** 2, 16), 4)
    return pd.DataFrame([part1, part2]).max()

def alpha072(data):
    """
    (rank(decay_linear(correlation(((high + low) / 2), adv40, 8.93345), 10.1519)) /
     rank(decay_linear(correlation(Ts_Rank(vwap, 3.72469), Ts_Rank(volume, 18.5188), 6.86671), 2.95011)))
    """
    adv40 = sum(data['volume'], 40) / 40
    return (rank(decay_linear(correlation((data['high'] + data['low']) / 2, adv40, 9), 10)) /
            rank(decay_linear(correlation(ts_rank(data['vwap'], 4), ts_rank(data['volume'], 19), 7), 3)))

def alpha073(data):
    """
    (max(rank(decay_linear(delta(vwap, 4.72775), 2.91864)),
     Ts_Rank(decay_linear(((delta(((open * 0.147155) + (low * (1 - 0.147155))), 2.03608) /
     ((open * 0.147155) + (low * (1 - 0.147155)))) * -1), 3.33829), 16.7411)) * -1)
    """
    part1 = rank(decay_linear(delta(data['vwap'], 5), 3))
    weighted = data['open'] * 0.147155 + data['low'] * 0.852845
    part2 = ts_rank(decay_linear(-1 * delta(weighted, 2) / weighted, 3), 17)
    return pd.DataFrame([part1, part2]).max() * -1

def alpha074(data):
    """
    ((rank(correlation(close, sum(adv30, 37.4843), 15.1365)) <
     rank(correlation(rank(((high * 0.0261661) + (vwap * (1 - 0.0261661)))), rank(volume), 11.4791))) * -1)
    """
    adv30 = sum(data['volume'], 30) / 30
    return ((rank(correlation(data['close'], sum(adv30, 37), 15)) <
             rank(correlation(rank(data['high'] * 0.0261661 + data['vwap'] * 0.9738339),
                             rank(data['volume']), 11))) * -1)

def alpha075(data):
    """
    (rank(correlation(vwap, volume, 4.24304)) < rank(correlation(rank(low), rank(adv50), 12.4413)))
    """
    adv50 = sum(data['volume'], 50) / 50
    return (rank(correlation(data['vwap'], data['volume'], 4)) <
            rank(correlation(rank(data['low']), rank(adv50), 12))).astype(int)

def alpha076(data):
    """
    (max(rank(decay_linear(delta(vwap, 1.24383), 11.8259)),
     Ts_Rank(decay_linear(Ts_Rank(correlation(IndNeutralize(low, IndClass.sector), adv81, 8.14941),
     19.569), 17.1543), 19.383)) * -1)
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv81 = sum(data['volume'], 81) / 81
    part1 = rank(decay_linear(delta(data['vwap'], 1), 12))
    part2 = ts_rank(decay_linear(ts_rank(correlation(data['low'], adv81, 8), 20), 17), 19)
    return pd.DataFrame([part1, part2]).max() * -1

def alpha077(data):
    """
    min(rank(decay_linear(((((high + low) / 2) + high) - (vwap + high)), 20.0451)),
     rank(decay_linear(correlation(((high + low) / 2), adv40, 3.1614), 5.64125)))
    """
    adv40 = sum(data['volume'], 40) / 40
    part1 = rank(decay_linear((data['high'] + data['low']) / 2 + data['high'] - data['vwap'] - data['high'], 20))
    part2 = rank(decay_linear(correlation((data['high'] + data['low']) / 2, adv40, 3), 6))
    return pd.DataFrame([part1, part2]).min()

def alpha078(data):
    """
    (rank(correlation(sum(((low * 0.352233) + (vwap * (1 - 0.352233))), 19.7428),
     sum(adv40, 19.7428), 6.83313))^rank(correlation(rank(vwap), rank(volume), 5.77492)))
    """
    adv40 = sum(data['volume'], 40) / 40
    weighted = data['low'] * 0.352233 + data['vwap'] * 0.647767
    return (rank(correlation(sum(weighted, 20), sum(adv40, 20), 7)) **
            rank(correlation(rank(data['vwap']), rank(data['volume']), 6)))

def alpha079(data):
    """
    (rank(delta(IndNeutralize(((close * 0.60733) + (open * (1 - 0.60733))), IndClass.sector), 1.23438)) <
     rank(correlation(Ts_Rank(vwap, 3.60973), Ts_Rank(adv150, 9.18637), 14.6644)))
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv150 = sum(data['volume'], 150) / 150
    return (rank(delta(data['close'] * 0.60733 + data['open'] * 0.39267, 1)) <
            rank(correlation(ts_rank(data['vwap'], 4), ts_rank(adv150, 9), 15))).astype(int)

def alpha080(data):
    """
    ((rank(Sign(delta(IndNeutralize(((open * 0.868128) + (high * (1 - 0.868128))), IndClass.industry),
     4.04545)))^Ts_Rank(correlation(high, adv10, 5.11456), 5.53756)) * -1)
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv10 = sum(data['volume'], 10) / 10
    return (rank(np.sign(delta(data['open'] * 0.868128 + data['high'] * 0.131872, 4))) **
            ts_rank(correlation(data['high'], adv10, 5), 6)) * -1

def alpha081(data):
    """
    ((rank(Log(product(rank((rank(correlation(vwap, sum(adv10, 49.6054), 8.47743))^4)), 14.9655))) <
     rank(correlation(rank(vwap), rank(volume), 5.07914))) * -1)
    """
    adv10 = sum(data['volume'], 10) / 10
    return ((rank(np.log(product(rank(rank(correlation(data['vwap'], sum(adv10, 50), 8)) ** 4), 15))) <
             rank(correlation(rank(data['vwap']), rank(data['volume']), 5))) * -1)

def alpha082(data):
    """
    (min(rank(decay_linear(delta(open, 1.46063), 14.8717)),
     Ts_Rank(decay_linear(correlation(IndNeutralize(volume, IndClass.sector),
     ((open * 0.634196) + (open * (1 - 0.634196))), 17.4842), 6.92131), 13.4283)) * -1)
    
    Note: Requires industry classification data
    """
    # Simplified version
    part1 = rank(decay_linear(delta(data['open'], 1), 15))
    part2 = ts_rank(decay_linear(correlation(data['volume'], data['open'], 17), 7), 13)
    return pd.DataFrame([part1, part2]).min() * -1

def alpha083(data):
    """
    ((rank(delay(((high - low) / (sum(close, 5) / 5)), 2)) * rank(rank(volume))) /
     (((high - low) / (sum(close, 5) / 5)) / (vwap - close)))
    """
    return (rank(delay((data['high'] - data['low']) / (sum(data['close'], 5) / 5), 2)) *
            rank(rank(data['volume'])) /
            ((data['high'] - data['low']) / (sum(data['close'], 5) / 5) / (data['vwap'] - data['close'])))

def alpha084(data):
    """
    SignedPower(Ts_Rank((vwap - ts_max(vwap, 15.3217)), 20.7127), delta(close, 4.96796))
    """
    return signedpower(ts_rank(data['vwap'] - ts_max(data['vwap'], 15), 21), delta(data['close'], 5))

def alpha085(data):
    """
    (rank(correlation(((high * 0.876703) + (close * (1 - 0.876703))), adv30, 9.61331))^
     rank(correlation(Ts_Rank(((high + low) / 2), 3.70596), Ts_Rank(volume, 10.1595), 7.11408)))
    """
    adv30 = sum(data['volume'], 30) / 30
    return (rank(correlation(data['high'] * 0.876703 + data['close'] * 0.123297, adv30, 10)) **
            rank(correlation(ts_rank((data['high'] + data['low']) / 2, 4), ts_rank(data['volume'], 10), 7)))

def alpha086(data):
    """
    ((Ts_Rank(correlation(close, sum(adv20, 14.7444), 6.00049), 20.4195) <
     rank(((open + close) - (vwap + open)))) * -1)
    """
    adv20 = sum(data['volume'], 20) / 20
    return ((ts_rank(correlation(data['close'], sum(adv20, 15), 6), 20) <
             rank(data['open'] + data['close'] - data['vwap'] - data['open'])) * -1)

def alpha087(data):
    """
    (max(rank(decay_linear(delta(((close * 0.369701) + (vwap * (1 - 0.369701))), 1.91233), 2.65461)),
     Ts_Rank(decay_linear(abs(correlation(IndNeutralize(adv81, IndClass.industry), close, 13.4132)),
     4.89768), 14.4535)) * -1)
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv81 = sum(data['volume'], 81) / 81
    part1 = rank(decay_linear(delta(data['close'] * 0.369701 + data['vwap'] * 0.630299, 2), 3))
    part2 = ts_rank(decay_linear(np.abs(correlation(adv81, data['close'], 13)), 5), 14)
    return pd.DataFrame([part1, part2]).max() * -1

def alpha088(data):
    """
    min(rank(decay_linear(((rank(open) + rank(low)) - (rank(high) + rank(close))), 8.06882)),
     Ts_Rank(decay_linear(correlation(Ts_Rank(close, 8.44728), Ts_Rank(adv60, 20.6966), 8.01266),
     6.65053), 2.61957))
    """
    adv60 = sum(data['volume'], 60) / 60
    part1 = rank(decay_linear(rank(data['open']) + rank(data['low']) - rank(data['high']) - rank(data['close']), 8))
    part2 = ts_rank(decay_linear(correlation(ts_rank(data['close'], 8), ts_rank(adv60, 21), 8), 7), 3)
    return pd.DataFrame([part1, part2]).min()

def alpha089(data):
    """
    (Ts_Rank(decay_linear(correlation(((low * 0.967285) + (low * (1 - 0.967285))), adv10, 6.94279),
     5.51607), 3.79744) - Ts_Rank(decay_linear(delta(IndNeutralize(vwap, IndClass.industry), 3.48158),
     10.1466), 15.3012))
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv10 = sum(data['volume'], 10) / 10
    return (ts_rank(decay_linear(correlation(data['low'], adv10, 7), 6), 4) -
            ts_rank(decay_linear(delta(data['vwap'], 3), 10), 15))

def alpha090(data):
    """
    ((rank((close - ts_max(close, 4.66719)))^Ts_Rank(correlation(IndNeutralize(adv40,
     IndClass.subindustry), low, 5.38375), 3.21856)) * -1)
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv40 = sum(data['volume'], 40) / 40
    return (rank(data['close'] - ts_max(data['close'], 5)) **
            ts_rank(correlation(adv40, data['low'], 5), 3)) * -1

def alpha091(data):
    """
    ((Ts_Rank(decay_linear(decay_linear(correlation(IndNeutralize(close, IndClass.industry), volume,
     9.74928), 16.398), 3.83219), 4.8667) - rank(decay_linear(correlation(vwap, adv30, 4.01303),
     2.6809))) * -1)
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv30 = sum(data['volume'], 30) / 30
    return (ts_rank(decay_linear(decay_linear(correlation(data['close'], data['volume'], 10), 16), 4), 5) -
            rank(decay_linear(correlation(data['vwap'], adv30, 4), 3))) * -1

def alpha092(data):
    """
    min(Ts_Rank(decay_linear(((((high + low) / 2) + close) < (low + open)), 14.7221), 18.8683),
     Ts_Rank(decay_linear(correlation(rank(low), rank(adv30), 7.58555), 6.94024), 6.80584))
    """
    adv30 = sum(data['volume'], 30) / 30
    part1 = ts_rank(decay_linear(((data['high'] + data['low']) / 2 + data['close']) < 
                                 (data['low'] + data['open']), 15), 19)
    part2 = ts_rank(decay_linear(correlation(rank(data['low']), rank(adv30), 8), 7), 7)
    return pd.DataFrame([part1, part2]).min()

def alpha093(data):
    """
    (Ts_Rank(decay_linear(correlation(IndNeutralize(vwap, IndClass.industry), adv81, 17.4193),
     19.848), 7.54455) / rank(decay_linear(delta(((close * 0.524434) + (vwap * (1 - 0.524434))),
     2.77377), 16.2664)))
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv81 = sum(data['volume'], 81) / 81
    return (ts_rank(decay_linear(correlation(data['vwap'], adv81, 17), 20), 8) /
            rank(decay_linear(delta(data['close'] * 0.524434 + data['vwap'] * 0.475566, 3), 16)))

def alpha094(data):
    """
    ((rank((vwap - ts_min(vwap, 11.5783)))^Ts_Rank(correlation(Ts_Rank(vwap, 19.6462),
     Ts_Rank(adv60, 4.02992), 18.0926), 2.70756)) * -1)
    """
    adv60 = sum(data['volume'], 60) / 60
    return (rank(data['vwap'] - ts_min(data['vwap'], 12)) **
            ts_rank(correlation(ts_rank(data['vwap'], 20), ts_rank(adv60, 4), 18), 3)) * -1

def alpha095(data):
    """
    (rank((open - ts_min(open, 12.4105))) < Ts_Rank((rank(correlation(sum(((high + low) / 2), 19.1351),
     sum(adv40, 19.1351), 12.8742))^5), 11.7584))
    """
    adv40 = sum(data['volume'], 40) / 40
    return (rank(data['open'] - ts_min(data['open'], 12)) <
            ts_rank(rank(correlation(sum((data['high'] + data['low']) / 2, 19),
                                   sum(adv40, 19), 13)) ** 5, 12)).astype(int)

def alpha096(data):
    """
    (max(Ts_Rank(decay_linear(correlation(rank(vwap), rank(volume), 3.83878), 4.16783), 8.38151),
     Ts_Rank(decay_linear(Ts_ArgMax(correlation(Ts_Rank(close, 7.45404), Ts_Rank(adv60, 4.13242),
     3.65459), 12.6556), 14.0365), 13.4143)) * -1)
    """
    adv60 = sum(data['volume'], 60) / 60
    part1 = ts_rank(decay_linear(correlation(rank(data['vwap']), rank(data['volume']), 4), 4), 8)
    part2 = ts_rank(decay_linear(ts_argmax(correlation(ts_rank(data['close'], 7),
                                                       ts_rank(adv60, 4), 4), 13), 14), 13)
    return pd.DataFrame([part1, part2]).max() * -1

def alpha097(data):
    """
    ((rank(decay_linear(delta(IndNeutralize(((low * 0.721001) + (vwap * (1 - 0.721001))),
     IndClass.industry), 3.3705), 20.4523)) - Ts_Rank(decay_linear(Ts_Rank(correlation(Ts_Rank(low,
     7.87871), Ts_Rank(adv60, 17.255), 4.97547), 18.5925), 15.7152), 6.71659)) * -1)
    
    Note: Requires industry classification data
    """
    # Simplified version
    adv60 = sum(data['volume'], 60) / 60
    return (rank(decay_linear(delta(data['low'] * 0.721001 + data['vwap'] * 0.278999, 3), 20)) -
            ts_rank(decay_linear(ts_rank(correlation(ts_rank(data['low'], 8),
                                                    ts_rank(adv60, 17), 5), 19), 16), 7)) * -1

def alpha098(data):
    """
    (rank(decay_linear(correlation(vwap, sum(adv5, 26.4719), 4.58418), 7.18088)) -
     rank(decay_linear(Ts_Rank(Ts_ArgMin(correlation(rank(open), rank(adv15), 20.8187), 8.62571),
     6.95668), 8.07206)))
    """
    adv5 = sum(data['volume'], 5) / 5
    adv15 = sum(data['volume'], 15) / 15
    return (rank(decay_linear(correlation(data['vwap'], sum(adv5, 26), 5), 7)) -
            rank(decay_linear(ts_rank(ts_argmin(correlation(rank(data['open']), rank(adv15), 21), 9),
                                     7), 8)))

def alpha099(data):
    """
    ((rank(correlation(sum(((high + low) / 2), 19.8975), sum(adv60, 19.8975), 8.8136)) <
     rank(correlation(low, volume, 6.28259))) * -1)
    """
    adv60 = sum(data['volume'], 60) / 60
    return ((rank(correlation(sum((data['high'] + data['low']) / 2, 20), sum(adv60, 20), 9)) <
             rank(correlation(data['low'], data['volume'], 6))) * -1)

def alpha100(data):
    """
    (0 - (1 * (((1.5 * scale(indneutralize(indneutralize(rank(((((close - low) - (high - close)) /
     (high - low)) * volume)), IndClass.subindustry), IndClass.subindustry))) -
     scale(indneutralize((correlation(close, rank(adv20), 5) - rank(ts_argmin(close, 30))),
     IndClass.subindustry))) * (volume / adv20))))
    
    Note: Requires industry classification data
    """
    # Simplified version without industry neutralization
    adv20 = sum(data['volume'], 20) / 20
    part1 = 1.5 * scale(rank(((data['close'] - data['low']) - (data['high'] - data['close'])) /
                            (data['high'] - data['low']) * data['volume']))
    part2 = scale(correlation(data['close'], rank(adv20), 5) - rank(ts_argmin(data['close'], 30)))
    return -1 * ((part1 - part2) * (data['volume'] / adv20))

def alpha101(data):
    """
    ((close - open) / ((high - low) + .001))
    """
    return (data['close'] - data['open']) / ((data['high'] - data['low']) + 0.001)

## Example Usage

```python
# Load your data
# df should have columns: open, high, low, close, volume, vwap
# and returns = (close / close.shift(1)) - 1

# Calculate a specific alpha
alpha1_values = alpha001(df)

# Calculate all alphas
alphas = {}
for i in range(1, 102):
    func_name = f'alpha{i:03d}'
    alphas[f'alpha_{i}'] = globals()[func_name](df)
```